# Values, types, and objects

Build the object and type mental model behind every Python data workflow, then apply it to a
small model-screening record without silently changing the source data.

**Lecture 1 · Python Foundations · CMOR 438 / INDE 577**


## How to use this notebook

**Estimated time:** 30 minutes core instruction, plus 20–30 minutes of practice and extension.

**Prerequisite:** complete notebook 00, select the **Rice DSM** kernel, and know how to run a cell.

The live route is marked **Core**. **Practice** sections ask you to produce or explain something.
**Extension** sections add nuance useful in production work but can be completed after class.

For every prediction prompt: pause, write down an answer, run the code, and explain any difference.
The goal is not to guess perfectly; it is to refine your model of how Python behaves.


## Learning objectives

By the end of this notebook, you should be able to:

- classify common scalar values and explain why type affects valid operations;
- trace arithmetic, comparison, Boolean, and conditional expressions;
- distinguish `None`, zero, an empty string, and `False` as different data states;
- explain assignment, equality, identity, mutability, and aliasing;
- normalize text without discarding the original input;
- diagnose suspicious values with `type`, `repr`, and small assertions; and
- justify a normalization or conversion policy instead of applying it silently.


## Why this matters in industry

A production model rarely receives a pristine Python object. Values arrive from CSV files, forms,
software interfaces (APIs—application programming interfaces), databases, and message queues. An
API is an agreed way for one program to request data or behavior from another; it is not necessarily
a website. A probability may arrive as the string `"0.873"`; a missing
label may be `None`, `""`, or a vendor-specific marker; and the same experiment identifier may contain
invisible whitespace or inconsistent case.

If we confuse representation with meaning, code can run successfully while corrupting a feature,
joining the wrong records, or routing a prediction incorrectly. Professional data work therefore
starts with three questions:

1. **What value did we receive?**
2. **What type and representation does it have now?**
3. **What explicit policy converts it into a trustworthy internal value?**


## Running scenario: one scientific model-screening record

Imagine a machine-learning service that screens spectroscopy experiments for an anomalous signal.
We will prepare three fields from one inference record:

- an experiment identifier containing extra whitespace and inconsistent case;
- a probability represented as text; and
- an optional scientist-review decision represented by an empty string.

We will preserve each raw value, derive a clean internal value, validate assumptions, and create a
human-readable status message. Later notebooks will represent many records with collections,
functions, classes, NumPy, and pandas.

For this instructional record, the service contract claims that the value is a calibrated
probability in `[0, 1]`. In real work, that claim requires evidence from calibration evaluation; a
bounded model score is not automatically a probability.


## Professional practice: two lenses, one record

The same three values raise different but complementary questions:

| Data scientist asks | Software engineer asks |
| --- | --- |
| What event does this experiment represent? | What input contract represents that event? |
| Is the score actually calibrated? | Where is `[0, 1]` validated? |
| Does missing review mean pending or not required? | Is missingness represented unambiguously? |
| What scientific cost follows a false positive? | Where is the decision threshold configured and tested? |
| Can normalization merge distinct experiments? | Are raw and normalized identifiers both traceable? |

We will not solve every production concern in one notebook. We will establish the habit of naming
assumptions, preserving evidence, validating the boundary, and writing checks that fail when the
contract is violated. Scientific validity and software reliability reinforce each other.


In [ ]:
raw_experiment_id = "  EXP-0042 "
raw_probability = "0.873"
raw_reviewer_decision = ""

print(repr(raw_experiment_id))
print(repr(raw_probability))
print(repr(raw_reviewer_decision))


## Core: values have types

A **value** is a piece of data. A **type** defines the operations Python supports for that value.
The scalar types we will use most often today are:

| Type | Example | Typical role |
| --- | --- | --- |
| `int` | `1_250` | counts and discrete quantities |
| `float` | `0.873` | measured values and model scores |
| `bool` | `True` | a two-state condition |
| `str` | `"EXP-0042"` | text and externally encoded fields |
| `NoneType` | `None` | an explicit absence of a value |

Python is dynamically typed: a name does not have one permanent type, but every object does. Use
`type(value)` when diagnosing a value; do not infer type from how printed output happens to look.


In [ ]:
observation_count = 1_250
prediction_probability = 0.873
is_reviewed = False
reviewer_name = None

print(type(observation_count), repr(observation_count))
print(type(prediction_probability), repr(prediction_probability))
print(type(is_reviewed), repr(is_reviewed))
print(type(reviewer_name), repr(reviewer_name))
print(type(raw_probability), repr(raw_probability))


### Practice: predict before running

Without executing the next cell, decide which expressions succeed and what type each successful
expression returns:

1. `3 + 2`
2. `3 / 2`
3. `"3" + "2"`
4. `"3" + 2`

For the failing expression, explain the mismatch in words before reading the captured error.


In [ ]:
print(3 + 2, type(3 + 2).__name__)
print(3 / 2, type(3 / 2).__name__)
print("3" + "2", type("3" + "2").__name__)

try:
    "3" + 2
except TypeError as error:
    print(f"Captured {type(error).__name__}: {error}")


## Core: conversion is an explicit boundary decision

External systems frequently encode numbers as text. `float(raw_probability)` asks Python to parse
the text and create a new floating-point object. It does not modify the original string.

Conversion can fail, so production code must decide what invalid input means: reject the record,
quarantine it, request correction, or apply a documented fallback. Silently replacing bad data with
zero is rarely a neutral choice.


In [ ]:
probability = float(raw_probability)

assert raw_probability == "0.873"  # raw evidence is unchanged
assert isinstance(probability, float)
assert 0.0 <= probability <= 1.0

print(repr(raw_probability), "->", probability)


### Extension: observe conversion failure safely

The next cell deliberately parses invalid input, but catches the exception so the notebook still
runs from top to bottom. An exception is data about a failed operation. Later we will design
functions that turn this signal into an accepted/rejected record workflow.


In [ ]:
invalid_probability = "not-recorded"

try:
    float(invalid_probability)
except ValueError as error:
    print(f"Rejected {invalid_probability!r}: {error}")


## Extension: floating-point values are approximations

Most decimal fractions cannot be represented exactly in binary floating-point. This is expected
computer arithmetic, not random Python behavior. Avoid exact equality for results of approximate
calculation; choose a tolerance justified by the domain.

For currency, regulated reporting, or other exact decimal policies, investigate `decimal.Decimal`
instead of assuming every `float` discrepancy can be rounded away.


In [ ]:
import math

computed = 0.1 + 0.2
print(f"stored result: {computed:.17f}")

assert computed != 0.3
assert math.isclose(computed, 0.3, rel_tol=1e-9, abs_tol=0.0)


## Core: absence is not the same as falseness

These values are all *falsy* in a Boolean context, but they do not carry the same meaning:

- `None`: no value is present;
- `0`: a numeric value is present and equals zero;
- `""`: text is present but empty; and
- `False`: a Boolean value is present and false.

Collapsing them with a convenient truthiness check can destroy information. Use `is None` when the
policy specifically concerns absence, and make empty-string handling explicit at the ingestion
boundary.


In [ ]:
missing_value = None
zero_value = 0
empty_text = ""
negative_decision = False

print(bool(missing_value), bool(zero_value), bool(empty_text), bool(negative_decision))

assert missing_value is None
assert zero_value is not None
assert empty_text is not None
assert negative_decision is not None


## Core: comparisons feed control flow

Comparisons such as `>=` and `==` produce Boolean objects. `and`, `or`, and `not` combine Boolean
conditions. An `if`/`elif`/`else` chain selects the first true branch.

Indentation is part of Python syntax. Order overlapping conditions from most restrictive to least
restrictive so an earlier branch does not capture a case intended for a later branch.


In [ ]:
if probability >= 0.90:
    confidence_band = "high"
elif probability >= 0.70:
    confidence_band = "medium"
else:
    confidence_band = "low"

assert confidence_band == "medium"
confidence_band


## Core: names refer to objects

Assignment binds a **name** to an **object**. It does not place a private copy of the object inside a
variable-shaped box. Two names can refer to the same object; this is called **aliasing**.

`==` asks whether values compare equal. `is` asks whether two references point to the identical
object. In ordinary application code, use identity primarily for singleton objects such as `None`.


In [ ]:
first_label = "medium"
second_label = confidence_band
same_value_built_separately = "".join(("med", "ium"))

assert first_label == same_value_built_separately
assert second_label == first_label

print("equal values:", first_label == same_value_built_separately)
print("identical objects:", first_label is same_value_built_separately)
print("missing reviewer:", reviewer_name is None)


## Core: mutability determines whether aliases observe change

Strings, integers, floats, Booleans, and tuples are immutable: their value cannot change in place.
Lists and dictionaries are mutable: an operation can change the existing object.

The small list below previews the next notebook. Because `alias_tags` and `model_tags` refer to the
same list, a mutation through either name is visible through both. Copying creates a separate list.


In [ ]:
model_tags = ["baseline"]
alias_tags = model_tags
copied_tags = model_tags.copy()

alias_tags.append("reviewed")

assert model_tags == ["baseline", "reviewed"]
assert copied_tags == ["baseline"]
assert alias_tags is model_tags
assert copied_tags is not model_tags


### Practice: trace the references

Before running the next cell, draw two objects and three name arrows for `source`, `alias`, and
`copy`. Predict both printed values. Then explain why changing one line to `copy = source` changes
the outcome.


In [ ]:
source = [10, 20]
alias = source
copy = source.copy()
source.append(30)

print("alias:", alias)
print("copy: ", copy)

assert alias == [10, 20, 30]
assert copy == [10, 20]


## Core: strings are immutable Unicode sequences

A Python `str` stores Unicode text. Indexing selects one code point and slicing selects a half-open
interval `[start:stop:step]`. String methods return new strings; they do not modify the original.

Text normalization is a policy decision. For an identifier documented as case-insensitive, trimming
outer whitespace and applying `casefold()` may be appropriate. A person's displayed name should not
automatically be forced through `.title()`—real names and writing systems do not follow one casing
rule. Preserve raw text whenever traceability matters.


In [ ]:
experiment_key = raw_experiment_id.strip().casefold()

assert raw_experiment_id == "  EXP-0042 "
assert experiment_key == "exp-0042"

print("raw:       ", repr(raw_experiment_id))
print("lookup key:", repr(experiment_key))


### Extension: visually identical Unicode can have different representations

Unicode text can represent the same human-readable character with different code-point sequences.
`unicodedata.normalize` can create a consistent representation when the data contract calls for it.
NFC is a conservative choice for canonical equivalence; stronger compatibility normalization can
change distinctions and should be adopted intentionally.


In [ ]:
import unicodedata

composed = "Café"
decomposed = "Café"

assert composed != decomposed
assert unicodedata.normalize("NFC", composed) == unicodedata.normalize(
    "NFC", decomposed
)

print(repr(composed), len(composed))
print(repr(decomposed), len(decomposed))


## Core: presentation should not change source values

F-strings create output text while leaving the underlying objects unchanged. Format specifications
control precision, percentages, alignment, and separators. Formatting `0.873` as `87.3%` is a
presentation choice; the calculation should continue using the original numeric probability.

The debugging form `{name=}` includes both the expression text and its representation, which is
useful in temporary diagnostics and logs.


In [ ]:
status_message = (
    f"experiment={experiment_key} | probability={probability:.1%} | "
    f"confidence={confidence_band}"
)

print(status_message)
print(f"debug: {raw_experiment_id=} {raw_probability=}")

assert probability == 0.873


## Worked example: prepare a trustworthy scoring decision

We now combine the notebook's decisions. The raw reviewer field is empty, which our hypothetical
contract defines as “not yet reviewed.” A medium-confidence prediction without review must be routed
to manual review.

Notice the layers:

1. preserve raw evidence;
2. convert and normalize into new internal values;
3. validate the internal values;
4. apply decision logic; and
5. format output separately.


In [ ]:
reviewer_decision = raw_reviewer_decision.strip().casefold()
has_human_approval = reviewer_decision == "approved"

if not 0.0 <= probability <= 1.0:
    raise ValueError("probability must be between 0 and 1")

requires_manual_review = confidence_band != "high" and not has_human_approval
route = "manual-review" if requires_manual_review else "automatic"

result_message = (
    f"{experiment_key}: route={route}, probability={probability:.3f}, "
    f"reviewed={has_human_approval}"
)

print(result_message)


In [ ]:
assert raw_experiment_id == "  EXP-0042 "
assert raw_probability == "0.873"
assert raw_reviewer_decision == ""
assert experiment_key == "exp-0042"
assert probability == 0.873
assert requires_manual_review is True
assert route == "manual-review"


## Debugging playbook

When a value behaves unexpectedly, avoid random conversions and installations. Inspect evidence in
this order:

1. `type(value)` — what kind of object is it?
2. `repr(value)` — are there quotes, escapes, or invisible whitespace?
3. the smallest failing expression — which operation has the wrong assumptions?
4. an assertion — what property must be true at this boundary?
5. the data contract — is the code wrong, or did the input violate the agreement?

`id(value)` can help demonstrate aliasing, but production logic should not depend on a memory
identity number. A diagnostic is not automatically a business rule.


## Practice: guided boundary cleanup

Prepare another record using this policy:

- experiment identifiers are case-insensitive and ignore surrounding whitespace;
- probabilities arrive as decimal text and must lie in `[0, 1]`;
- an absent reviewer is represented by `None`, not by an empty string; and
- probabilities below `0.80` require review.

Complete the derivation, then use the assertions as success criteria.


In [ ]:
practice_raw_experiment = " Sample-17  "
practice_raw_probability = "0.74"
practice_reviewer = None

practice_experiment_key = practice_raw_experiment.strip().casefold()
practice_probability = float(practice_raw_probability)
practice_requires_review = (
    practice_probability < 0.80 and practice_reviewer is None
)

assert practice_experiment_key == "sample-17"
assert math.isclose(practice_probability, 0.74)
assert practice_requires_review is True


## Practice: independent feature-flag parsing

An instrument's software interface—its API—sends `" YES "`, `"no"`, or an empty string for a
feature flag. The instrument is the producer; our Python code is the caller that must interpret its
documented output contract. See [What is an API?](../../supplementary-materials/computing-foundations/07-what-is-an-api.md)
for the full distinction.

Create `normalized_flag` and then assign `feature_enabled` according to this contract:

- `yes` means `True`;
- `no` means `False`; and
- an empty value means `None` because no decision was supplied.

Do not use `bool(raw_flag)`: any nonempty string, including `"no"`, is truthy. Your code should pass
the assertions below.


In [ ]:
raw_flag = " YES "
normalized_flag = raw_flag.strip().casefold()

if normalized_flag == "yes":
    feature_enabled = True
elif normalized_flag == "no":
    feature_enabled = False
elif normalized_flag == "":
    feature_enabled = None
else:
    raise ValueError(f"unrecognized feature flag: {raw_flag!r}")

assert normalized_flag == "yes"
assert feature_enabled is True


## Extension: make and defend a policy

Suppose experiment identifiers sometimes contain internal spaces, hyphens, underscores, and accented
letters. A teammate proposes deleting every non-ASCII character before joining records.

Write a short response that addresses:

1. what information the proposal could destroy;
2. which upstream data contract you would request;
3. which transformations are reversible;
4. how you would measure records affected by the policy; and
5. which raw and normalized fields you would retain for auditability.

There is no universal normalization rule. A strong answer identifies assumptions and evidence
rather than choosing the shortest string method chain.


## Common failure modes

| Symptom | Likely mistake | Better diagnostic or policy |
| --- | --- | --- |
| `"10" + "5"` becomes `"105"` | text was never converted | inspect `type`, then parse at the boundary |
| `bool("no")` is `True` | truthiness was confused with domain meaning | enumerate accepted text values |
| valid zero treated as missing | `if not value` collapsed distinct states | check `value is None` when absence matters |
| aliases change together | two names reference one mutable object | copy deliberately or document shared mutation |
| names/IDs change unexpectedly | aggressive normalization policy | preserve raw text and specify a normalized key |
| decimal equality surprises | binary floating-point approximation | compare with a justified tolerance |

Avoid “fixing” these symptoms with scattered casts or string replacements. Put conversion and
validation at a named boundary so the rest of the program receives trustworthy values.


## Retrieval practice

Answer without running code:

1. What is the difference between a name and an object?
2. When should `is` be preferred to `==`?
3. Why can `None`, `0`, `""`, and `False` not be treated as interchangeable?
4. Why does mutating an alias affect the original list?
5. What evidence does `repr` reveal that `print` may hide?
6. Why should raw text usually be preserved after normalization?
7. What makes a conversion policy a business decision rather than merely Python syntax?


## Takeaway and next step

Python executes operations on typed objects; names let us refer to those objects. Reliable data work
keeps raw evidence, performs explicit conversions, validates boundaries, distinguishes absence from
falseness, and treats text normalization as policy.

Next, notebook 02 develops strings as immutable Unicode sequences and treats text parsing,
normalization, validation, and formatting as explicit data-boundary decisions.


## Further reading

- [Python tutorial: an informal introduction](https://docs.python.org/3.12/tutorial/introduction.html)
- [Python standard types](https://docs.python.org/3.12/library/stdtypes.html)
- [Python floating-point arithmetic](https://docs.python.org/3.12/tutorial/floatingpoint.html)
- [Python Unicode HOWTO](https://docs.python.org/3.12/howto/unicode.html)
- [PEP 8: Style Guide for Python Code](https://peps.python.org/pep-0008/)

The Python tutorial assumes some prior programming experience. This notebook supplies additional
mental models, diagnostics, and data-science context for students learning both at once.
